In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("../data/FULL_INSECT_DATA.csv")
df.head()

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import top_k_accuracy_score

df["day_of_year"] = pd.to_datetime(df["eventDate"]).dt.dayofyear
df["hour"] = pd.to_datetime(df["eventDate"]).dt.hour


species_counts = df["species"].value_counts()
viable_species = species_counts[species_counts >= 200].index
df_viable = df[df["species"].isin(viable_species)].copy()
print(f"{len(viable_species)} species kept, {len(df_viable)} rows")


eco_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df_viable["ecoregion_enc"] = eco_encoder.fit_transform(df_viable[["ecoregion"]])

precip_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df_viable["precip_type_enc"] = precip_encoder.fit_transform(df_viable[["precip_type"]])

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_viable["species"])


df_viable["day_sin"] = np.sin(2 * np.pi * df_viable["day_of_year"] / 365.25)
df_viable["day_cos"] = np.cos(2 * np.pi * df_viable["day_of_year"] / 365.25)

df_viable["hour_sin"] = np.sin(2 * np.pi * df_viable["hour"] / 24.0)
df_viable["hour_cos"] = np.cos(2 * np.pi * df_viable["hour"] / 24.0)


params = [
    "hourly_temp_C", "hourly_humidity_percent", "soil_temp_C",
    "cloud_cover_pct", "snow_depth_m", "precip_type_enc",
    "hour_sin", "hour_cos", "day_sin", "day_cos",
    "ecoregion_enc", "land_cover_code"
]
X = df_viable[params]


groups = (df_viable["decimalLatitude"].round(3).astype(str)
          + "_" + df_viable["decimalLongitude"].round(3).astype(str))
gss = GroupShuffleSplit(test_size=0.2, random_state=67)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]


model = RandomForestClassifier(
    n_estimators=100,
    min_samples_leaf=10,
    max_features="sqrt",
    oob_score=False,
    n_jobs=2,          
    random_state=67
)
model.fit(X_train, y_train)




batch_size = 5000
total_samples = X_train.shape[0]
correct_predictions = 0

print(f"Evaluating training accuracy across {total_samples} samples...")


for i in range(0, total_samples, batch_size):
    X_batch = X_train[i : i + batch_size]
    y_batch = y_train[i : i + batch_size]

   
    batch_preds = model.predict(X_batch)


    correct_predictions += np.sum(batch_preds == y_batch)

train_accuracy = (correct_predictions / total_samples) * 100
print(f"\nSuccess! Training Accuracy: {train_accuracy:.2f}%")

print(f"Test accuracy:  {model.score(X_test, y_test):.4f}")

test_probs = model.predict_proba(X_test)
top20_acc = top_k_accuracy_score(y_test, test_probs, k=20, labels=model.classes_)
print(f"Top-20 test accuracy: {top20_acc:.4f}")

importances = pd.Series(model.feature_importances_, index=params).sort_values(ascending=False)
print(importances)


In [ ]:
batch_size = 5000
total_samples = X_train.shape[0]
correct_predictions = 0


for i in range(0, total_samples, batch_size):
    X_batch = X_train[i : i + batch_size]
    y_batch = y_train[i : i + batch_size]

    batch_preds = model.predict(X_batch)
    correct_predictions += np.sum(batch_preds == y_batch)


train_accuracy = (correct_predictions / total_samples) * 100
print(f"Training Accuracy: {train_accuracy:.2f}%")

print(f"Test accuracy:  {model.score(X_test, y_test):.4f}")

test_probs = model.predict_proba(X_test)
top20_acc = top_k_accuracy_score(y_test, test_probs, k=20, labels=model.classes_)
print(f"Top-20 test accuracy: {top20_acc:.4f}")

importances = pd.Series(model.feature_importances_, index=params).sort_values(ascending=False)
print(importances)

In [ ]:
import joblib

joblib.dump(model, "rf_model.pkl")

joblib.dump(eco_encoder, "eco_encoder.pkl")
joblib.dump(precip_encoder, "precip_encoder.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

lightgbm model:

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import time
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit, train_test_split


df = pd.read_csv("../data/FULL_INSECT_DATA.csv")

df["day_of_year"] = pd.to_datetime(df["eventDate"]).dt.dayofyear
df["hour"] = pd.to_datetime(df["eventDate"]).dt.hour
df["day_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
df["day_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)


df["location_group"] = (df["decimalLatitude"].round(3).astype(str)
                         + "_" + df["decimalLongitude"].round(3).astype(str))


species_counts = df["species"].value_counts()
viable_by_count = species_counts[species_counts >= 200].index


species_spread = df.groupby("species")["location_group"].nunique()
viable_by_spread = species_spread[species_spread >= 10].index

viable_species = set(viable_by_count) & set(viable_by_spread)
df_viable = df[df["species"].isin(viable_species)].copy()


label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_viable["species"])

for col in ["ecoregion", "land_cover_code", "precip_type"]:
    df_viable[col] = df_viable[col].astype("category")

params = ["hourly_temp_C", "hourly_humidity_percent", "soil_temp_C",
          "cloud_cover_pct", "snow_depth_m", "precip_type",
          "hour_sin", "hour_cos", "day_sin", "day_cos",
          "ecoregion", "land_cover_code"]
X = df_viable[params]
cat_features = ["precip_type", "ecoregion", "land_cover_code"]


gss_test = GroupShuffleSplit(test_size=0.2, random_state=67)
train_idx, test_idx = next(gss_test.split(X, y, groups=df_viable["location_group"]))

X_train_full, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train_full, y_test = y[train_idx], y[test_idx]
groups_train_full = df_viable["location_group"].iloc[train_idx]


gss_val = GroupShuffleSplit(test_size=0.1, random_state=67)
tr_idx, val_idx = next(gss_val.split(X_train_full, y_train_full, groups=groups_train_full))

X_tr, X_val = X_train_full.iloc[tr_idx], X_train_full.iloc[val_idx]
y_tr, y_val = y_train_full[tr_idx], y_train_full[val_idx]

'''
calibration_model = lgb.LGBMClassifier(
    objective="multiclass", 
    num_class=len(label_encoder.classes_),
    learning_rate=0.03,      
    n_estimators=10, 
    num_leaves=63, 
    min_child_samples=50,      
    verbose=-1, 
    n_jobs=-1, 
    random_state=67
)
start = time.time()
calibration_model.fit(X_tr, y_tr, categorical_feature=cat_features)
elapsed = time.time() - start
print(f"roughly {elapsed * 30:.0f}s for 300 rounds")
'''

model = lgb.LGBMClassifier(
    objective="multiclassova",     
    num_class=len(label_encoder.classes_),
    learning_rate=0.03,
    n_estimators=500,

    max_depth=6, 
    num_leaves=31,
    min_child_samples=100,
    
    colsample_bytree=0.5,
    subsample=0.7,
    bagging_freq=1,
    
    reg_lambda=10.0,
    
    verbose=-1, 
    n_jobs=-1, 
    random_state=67
)

model.fit(
    X_tr, y_tr, 
    eval_set=[(X_val, y_val)],
    categorical_feature=cat_features,
    callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=True)] 
)

print(f"Stopped at round: {model.best_iteration_}")
print(f"Train accuracy: {model.score(X_tr, y_tr):.4f}")
print(f"Test accuracy:  {model.score(X_test, y_test):.4f}")

def chunked_top_k_accuracy(model, X_test, y_test, k=20, chunk_size=20_000):
    correct = 0
    for start in range(0, len(X_test), chunk_size):
        end = start + chunk_size
        probs = model.predict_proba(X_test.iloc[start:end]).astype(np.float32)
        top_k = np.argsort(probs, axis=1)[:, -k:]
        batch_y = y_test[start:end]
        correct += sum(batch_y[i] in top_k[i] for i in range(len(batch_y)))
    return correct / len(X_test)

top20_acc = chunked_top_k_accuracy(model, X_test, y_test, k=20)
print(f"Top 20 test accuracy: {top20_acc:.4f}")


In [ ]:
joblib.dump(model, "gb_model.pkl")

joblib.dump(label_encoder, "gb_label_encoder.pkl")

In [ ]:
cat_categories = {col: df_viable[col].cat.categories.tolist() for col in cat_features}
joblib.dump(cat_categories, "gb_cat_categories.pkl")